# Version 2 — a walkthrough of the solver

This notebook walks through *how the v2 solver works*, section by section, for a
reader who is comfortable with code and maths but new to this project. It does
**not** reimplement anything — it imports the tested engine (`v2_full_game.py`)
and calls it piece by piece, explaining each part as we go. The engine is the
single source of truth; this notebook is the guided tour.

**What the game is, in three sentences.** A trader must convert £100,000 into
dollars over four rounds, selling to a market maker who can see the true
exchange rate while the trader cannot. Each round the trader makes a few priced
offers (learning from each accept/reject), then any leftover pounds carry to the
next round; at the end there are penalties for unconverted pounds and for
missing a dollar target. We want the *optimal* trading strategy and what it is
worth.

**How we solve it.** By *backward induction* — computing the value of every
situation by working backwards from the end of the game. This is exact as a
method, but computed on grids, so the answer is a close approximation with a
measured error (~0.1%). Its purpose is to be a trustworthy benchmark for v3
(which adds competing traders and needs machine learning).

We proceed in the order the engine is built:
1. the trader/game definition,
2. the grids (how continuous quantities are stored),
3. the terminal reward (where backward induction starts),
4. solving one round,
5. chaining the rounds into the full game,
6. reading out and drawing the optimal strategy,
7. the validation gates that prove it works.

## Setup

We import the engine. Everything below calls into it. For anything that
actually solves the game, we use a deliberately **coarse, fast grid** so cells
run in seconds rather than minutes — the *shape* of every result is the same as
at full resolution, only the last couple of digits differ.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import v2_full_game as v2

# a fast, coarse grid for the live demos (full resolution is slower)
FAST = dict(n_offer=25, n_a=9)
print("engine loaded.")

engine loaded.


## 1. The trader and the game (`TraderSpec`)

Everything about one instance of the game lives in a single object,
`TraderSpec`. Its fields are the whole rulebook:

- `L` — starting pounds (£100,000)
- `T` — the dollar target ($125,000)
- `A` — penalty on any pounds left unconverted (2%)
- `B` — penalty on any shortfall below the dollar target (3%)
- `rounds` — number of trading rounds (4)
- `K` — offers allowed per round (3)
- `params` — the rate process: starting rate `a0`, volatility `sd`, and the
  Bureau de Change fee.

Every field has a default (Team 1's card), and every field is changeable — so
you can solve the game for any team by passing different numbers. Nothing about
the rules is hard-coded anywhere else.

In [2]:
spec = v2.TraderSpec()          # Team 1's card (the defaults)
print("L =", spec.L, " T =", spec.T, " A =", spec.A, " B =", spec.B)
print("rounds =", spec.rounds, " K =", spec.K)
print("rate: a0 =", spec.params.a0, " sd =", spec.params.sd,
      " BdC fee =", spec.params.bdc_fee)

# a different team is just different numbers:
big = v2.TraderSpec(L=500_000, T=625_000)
print("\na bigger team:  L =", big.L, " T =", big.T)

L = 100000.0  T = 125000.0  A = 0.02  B = 0.03
rounds = 4  K = 3
rate: a0 = 1.25  sd = 0.05  BdC fee = 0.02

a bigger team:  L = 500000  T = 625000


## 2. The grids (`Grids`)

The value of a situation depends on continuous quantities — how many pounds you
hold, how many dollars, and the last rate you saw. A computer can't store a
value for *every* real number, so it stores values at a finite set of sample
points (a **grid**) and interpolates in between. The `Grids` object builds all
of these.

Key point that trips people up: **there are two kinds of grid.**

- *State grids* (pounds, dollars, anchor) — where the value function is
  **stored**. These are unavoidable (you must sample a continuous function
  somewhere) and accurate because the value is smooth in these directions.
- *Action grids* (offer price, offer size) — the **menu of choices** the trader
  picks from.

Let's build one and look at the grids.

In [3]:
g = v2.Grids(spec, **FAST)

print("pounds grid (state):", g.n_c, "points from 0 to", f"{g.c[-1]:,.0f}",
      "-> spacing", f"{g.dc:,.0f}")
print("dollar grid (state):", g.n_d, "points, spacing", f"{g.dd:,.0f}")
print("   target", f"${spec.T:,.0f}", "sits on node", g.jT,
      "=", f"${g.d[g.jT]:,.0f}", "(exact, so the penalty 'kink' isn't smeared)")
print("anchor grid (state):", g.n_a, "points from",
      f"{g.a[0]:.3f}", "to", f"{g.a[-1]:.3f}")
print("offer-price grid (action):", g.n_offer, "prices, in std-devs from the anchor")
print("sell-fraction grid (action):", g.f_pos.size,
      "fractions of holdings (so any purse size gets the same choices)")

pounds grid (state): 17 points from 0 to 100,000 -> spacing 6,250
dollar grid (state): 41 points, spacing 4,630
   target $125,000 sits on node 27 = $125,000 (exact, so the penalty 'kink' isn't smeared)
anchor grid (state): 9 points from 0.850 to 1.650
offer-price grid (action): 25 prices, in std-devs from the anchor
sell-fraction grid (action): 25 fractions of holdings (so any purse size gets the same choices)


### Why dollars gets more points than pounds

The value is a straight line in *pounds* (nothing special happens at any pound
level), so a coarse grid captures it perfectly. But in *dollars* there is a
sharp **kink** at the target `T` — the deficit penalty switches on there — so
the dollar grid is finer *and* deliberately places a node exactly on `T` so the
corner isn't blurred. Resolution should match how much the function bends:
smooth directions coarse, the kink direction fine.

The grids also carry a **quadrature** — a set of rate sample points and their
probabilities — used to take averages over the unknown rate quickly. That's the
`rate_samples` / `rate_weights` / `cumulative_mass` machinery; you don't need to
read it to follow the solver, just know it means "take an expectation over the
rate, fast".

## 3. Where backward induction starts: the terminal reward

Backward induction works from the end backwards, so it needs a starting point:
the value of *finishing* the game. After the last round the trader holds some
pounds `c` and dollars `d`, a final settlement rate is drawn, and the score is
just the rulebook formula — pounds penalised by `A`, dollars converted with a
shortfall penalty `B`.

Because the settlement rate isn't known when the last decisions are made, the
recursion uses the *expected* version, replacing `1/rate` with its average,
called `kappa`. Because `1/x` is curved, this average comes out slightly above
`1/a` — a small but real effect.

In [4]:
# kappa = E[1/settlement_rate | anchor]. Slightly above 1/anchor.
for a in (1.15, 1.25, 1.35):
    print(f"anchor {a}:  kappa = {v2.kappa(g, a, spec.params.sd):.5f}"
          f"   vs 1/anchor = {1/a:.5f}")

anchor 1.15:  kappa = 0.87122   vs 1/anchor = 0.86957
anchor 1.25:  kappa = 0.80129   vs 1/anchor = 0.80000
anchor 1.35:  kappa = 0.74176   vs 1/anchor = 0.74074


## 4. Solving one round (`solve_round`)

This is the heart. Given the value of *entering the next round*, `solve_round`
computes the value of entering *this* round, by reasoning through the round's
offers.

The logic of one offer: the trader picks a price `P` and size `q`. The offer is
accepted with some probability (then pounds fall, dollars rise, and the trader
learns the true rate is *above* `P`) or rejected (position unchanged, and the
trader learns the rate is *below* `P`). The value of the offer is the
probability-weighted average of these two outcomes — and the trader picks the
best `P` and `q`, or stops and uses the Bureau de Change.

Within a round the trader's **belief** about the rate is an interval `(low,
high)` — a floor from acceptances, a ceiling from rejections — that narrows with
each offer.

Let's solve the last round (its "next round" is just the settlement layer) and
look at what it produces.

In [5]:
# the terminal layer at every anchor node = the reward the last round works from
Wnext = np.stack([v2.terminal_plane(spec, g, a) for a in g.a])

# solve one round at the starting anchor
entry_value, U_by_offers, _ = v2.solve_round(spec, g, Wnext, spec.params.a0)

# entry_value is the value of ENTERING this round, for every (pounds, dollars).
# Value of entering holding the full book and no dollars yet:
full_book = entry_value[g.n_c - 1, 0]
print(f"value of entering the last round with the full £{spec.L:,.0f}, $0:")
print(f"   £{full_book:,.0f}")

value of entering the last round with the full £100,000, $0:
   £98,731


## 5. Chaining the rounds into the full game (`solve_v2`)

`solve_round` does one round; `solve_v2` chains them. It seeds the recursion
with the settlement layer, then solves round 4, then 3, then 2 — each using the
already-solved next round as its reward — and finally round 1 at the single
known starting rate. The value at the true start (full book, no dollars) is the
**headline number**: the best expected final wealth, hence the expected
profit/loss.

This is the whole solve. On the coarse grid it takes a few seconds.

In [6]:
sol = v2.solve_v2(spec, g)          # the full backward induction
print(f"E[final wealth] = £{sol.value:,.0f}")
print(f"expected P/L    = {sol.pl()*100:+.2f}%")
print("\n(the loss is expected: the target equals the fair value of the book and")
print(" the rate has no drift, so there's no edge to win -- only fees/penalties.)")

E[final wealth] = £99,773
expected P/L    = -0.23%

(the loss is expected: the target equals the fair value of the book and
 the rate has no drift, so there's no edge to win -- only fees/penalties.)


## 6. The main output: the optimal strategy, drawn

The solve gives us the optimal *policy* — what to do in every situation. But the
strategy is *contingent*: what the trader actually does depends on which rates
the game happens to draw. So "the optimal path" is shown for a specific rate
draw. `play_game` plays the optimal policy on one draw and records every move;
`print_game_trace` prints it; `plot_game` draws it.

Below we play a representative game and show the optimal path — you'll see the
signature behaviour: tiny **probe** offers to locate the rate, then a big
**commit**, carrying pounds across rounds when a round is unfavourable.

In [7]:
# play the optimal policy on some rates; here we let it pick a representative game
res = v2.simulate_paths(sol, n_paths=800, seed=1)
trace = v2._pick_representative_game(sol, res)

v2.print_game_trace(trace)

  anchor   hidden X      offer            verdict     -> pounds left   dollars banked
------------------------------------------------------------------------------
Round 1: a=1.2500  X=1.2331 (z=-0.34)
  offer 1: P=1.2667 (z=+0.33)  sell GBP      200   reject   -> c= 100,000   d=$         0
  offer 2: P=1.2500 (z=+0.00)  sell GBP  100,000   reject   -> c= 100,000   d=$         0
  offer 3: P=1.2333 (z=-0.33)  sell GBP  100,000   reject   -> c= 100,000   d=$         0
  BdC: carry (no dump)                             -> c= 100,000   d=$         0
------------------------------------------------------------------------------
Round 2: a=1.2331  X=1.1121 (z=-2.42)
  offer 1: P=1.2497 (z=+0.33)  sell GBP      200   reject   -> c= 100,000   d=$         0
  offer 2: P=1.2331 (z=+0.00)  sell GBP  100,000   reject   -> c= 100,000   d=$         0
  offer 3: P=1.2164 (z=-0.33)  sell GBP  100,000   reject   -> c= 100,000   d=$         0
  BdC: carry (no dump)                             -> c= 10

In [8]:
# draw that same optimal path
v2.plot_game(trace, title=f"Optimal strategy  |  P/L = {trace['pl']*100:+.2f}%")
plt.show()

To see the optimal play on a **specific** sequence of rates, pass them in:

```python
sol, trace = v2.optimal_strategy(spec, rate_path=[1.20, 1.24, 1.19, 1.31],
                                 grid=v2.Grids(spec, **FAST))
```

`optimal_strategy` is the one-call version of everything above: give it a
trader, it solves the game and draws the optimal path.

## 7. How good is it? The clairvoyant benchmark

We compare optimal play against a **clairvoyant** who knows the entire rate path
in advance and sells at the best moment. The clairvoyant is an upper bound
nobody can legitimately reach; the gap between it and optimal play is the
**regret** — the price of trading without foresight. This regret is the number
v3's learning agents will ultimately be measured against.

In [9]:
Wcv = v2.clairvoyant_wealth(spec, res["X"], res["a5"])
regret = (Wcv - res["W"]).mean()
print(f"optimal play (simulated):  £{res['W'].mean():,.0f}  "
      f"({(res['W'].mean()-spec.L)/spec.L*100:+.2f}%)")
print(f"clairvoyant (knows rates): £{Wcv.mean():,.0f}  "
      f"({(Wcv.mean()-spec.L)/spec.L*100:+.2f}%)")
print(f"regret (the value of foresight): £{regret:,.0f} per game")

optimal play (simulated):  £99,599  (-0.40%)
clairvoyant (knows rates): £104,110  (+4.11%)
regret (the value of foresight): £4,510 per game


## 8. Why we trust it: the validation gates

The method is checked by four runnable tests. Two of them are quick enough to
run here:

- **Gate 1** restricts the full solver to a single round with all-or-nothing
  offers and checks it reproduces the earlier, separately-validated v1 solver
  (and v0's hand-derived formula) to six digits. This is the closest thing to a
  proof of correctness: the machinery reproduces an answer derived a completely
  different way.
- **Gate 2** switches the deficit penalty off and checks the value becomes
  exactly linear — confirming the kink is the only nonlinearity and the grids
  add no spurious error.

(The other two — a 20,000-game simulation matching the claimed value, and money
conservation — take longer and live in the test suite.)

In [10]:
print("Gate 1 (reduces to the validated v1 solver):")
v2.gate1_v1_anchor(K=3, verbose=True)

Gate 1 (reduces to the validated v1 solver):


  K=1: v2 1.225337  v1 1.225337  gap 1.84e-07
  K=2: v2 1.227245  v1 1.227249  gap 4.22e-06
  K=3: v2 1.229742  v1 1.229748  gap 5.12e-06
  K=1 vs v0 closed form: gap 1.95e-07


np.float64(5.121754266035339e-06)

In [11]:
print("Gate 2 (penalty off -> value exactly linear):")
v2.gate2_b0_linearity(verbose=True)

Gate 2 (penalty off -> value exactly linear):


  (i) linearity with probes re-admitted (rel to L): 2.74e-14


  (ii) d-linearity under the q>0 ban (rel to L): 4.63e-15
       d-slope vs independent kappa smoothing (rel): 1.92e-15
  (iii) c-curvature the ban creates (rel to L): 9.46e-16  -- continuous q keeps the action set scale-free, so the ban adds no curvature


(2.735760062932968e-14, np.float64(1.918479474997995e-15))

## Recap

- The **game** is defined entirely by `TraderSpec`.
- The **grids** store the continuous value function at sample points; state
  grids are unavoidable and accurate, action grids are the menu of choices.
- **Backward induction** starts from the settlement reward and works backwards;
  `solve_round` does one round, `solve_v2` chains them.
- The **optimal strategy** is read out by `play_game` / `optimal_strategy` and
  drawn as the path it follows on a given rate draw.
- The **clairvoyant regret** measures how much foresight would be worth — the
  benchmark v3 inherits.
- The **gates** prove the implementation is correct (Gate 1 against independent
  answers) and the structure is as claimed (Gate 2).

The engine (`v2_full_game.py`) is terse on purpose — it's the precise,
verifiable computation. This notebook is the readable tour on top of it. For the
full-resolution numbers, drop the `FAST` grid and call `v2.solve_v2(spec)`
directly (slower, tighter last digits).